[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239232-lesson-6-agent)

# Agent

## 回顾

我们构建了一个 Router（路由器）。

* 聊天模型会根据用户输入决定是否进行工具调用
* 我们使用条件边来路由到将调用工具的节点，或者直接结束

![Screenshot 2024-08-21 at 12.44.33 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac0ba0bd34b541c448cc_agent1.png)

## 目标

现在，我们可以将其扩展为一个通用 Agent 架构。

在上面的 Router 中，我们调用了模型，如果它选择调用工具，我们会向用户返回一个 `ToolMessage`。

但是，如果我们直接将 `ToolMessage` *传回给模型*，会怎样呢？

我们可以让模型要么 (1) 调用另一个工具，要么 (2) 直接做出响应。

这就是 [ReAct](https://react-lm.github.io/) 背后的直觉，一种通用的 Agent 架构。

* `act` —— 让模型调用特定工具
* `observe` —— 将工具输出传回给模型
* `reason` —— 让模型根据工具输出推理下一步该做什么（例如，调用另一个工具或直接响应）

这种[通用架构](https://blog.langchain.com/planning-for-agents/)可以应用于多种类型的工具。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_deepseek langchain_core langgraph langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

这里，我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

我们将日志记录到一个名为 `langchain-academy` 的项目中。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
from langchain_deepseek import ChatDeepSeek

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatDeepSeek(model="deepseek-chat")

# 在此 ipynb 中，我们将并行工具调用设置为 false，因为数学运算通常是顺序执行的，且这次我们有 3 个可以做数学的工具
# DeepSeek 模型默认会使用并行工具调用来提高效率，参见 https://python.langchain.com/docs/how_to/tool_calling_parallel/
# 可以尝试调整它，观察模型在处理数学方程时的行为！
llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)

让我们创建 LLM 并通过 system prompt 设定期望的 Agent 行为。

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

和之前一样，我们使用 `MessagesState` 并定义一个包含工具列表的 `Tools` 节点。

`Assistant` 节点就是绑定了工具的模型。

我们创建一个包含 `Assistant` 和 `Tools` 节点的图。

添加 `tools_condition` 边，它会根据 `Assistant` 是否调用了工具，将控制流路由到 `End` 或 `Tools`。

现在，我们添加一个新步骤：

我们将 `Tools` 节点连接*回* `Assistant`，形成一个循环。

* 在 `assistant` 节点执行后，`tools_condition` 检查模型的输出是否为工具调用。
* 如果是工具调用，流程会导向 `tools` 节点。
* `tools` 节点连接回 `assistant`。
* 只要模型决定调用工具，这个循环就会持续。
* 如果模型响应不是工具调用，流程将导向 END，终止整个过程。

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些节点执行实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些决定控制流的走向
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# 显示
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 5")]
messages = react_graph.invoke({"messages": messages})

In [ ]:
for m in messages['messages']:
    m.pretty_print()

## LangSmith

我们可以在 LangSmith 中查看追踪记录。